In [3]:
# Lista 8 Oliwia Borkowska
import random
import time
import math
from typing import List
# Paramtery algorytmu
liczba_osobnikow = 10 # k - liczba osobników w populacji
liczba_genow= 10 # r – liczba genów (x nalezy od 0 do 1023)
maksymalnie_pokolen = 50
prawdopodbienstwo_krzyzowania = 0.5
prawdopodobienstwo_mutacji = 0.01
epsilon = 0.001 # tolerancja do warunku stopu
# c = chromosom = osobnik

In [64]:
# (0) KODOWANIE - NATURALNE KODOWANIE BINARNE

"""dekodowanie osobnika c (chromosomu binarnego) do zmiennej optymalizacyjnej x. Naturalne kodowanie binarne: Wartości genów osobnika reprezentują liczbę naturalną zakodowaną binarnie. x = sum(c(i) * 2^(r-i-1))"""

def decode_chromosom(chromosom: List[int]) -> int:

    x = 0
    r = len(chromosom)
    for i in range(r):
        x += chromosom[i] * (2 ** (r - i - 1))
    return x

In [65]:
# FUNKCJA CELU (funkcja przystosowania - fitness function)
# funkcja celu (przystosowania): f(x) = x * sin(x)

def funkcja_przystosowania(x: int) -> float:
    return x * math.sin(x)

In [2]:
# (1) WYBÓR POPULACJI POCZĄTKOWEJ

""" Losowy, polegający na przeprowadzeniu 𝑘 ∙ 𝑟 losowań wartości ze zbioru {0,1}. jednakowym prawdopodobieństwem"""

def populacja_poczatkowa() -> List[List[int]]:
    populacja = []
    for _ in range(liczba_osobnikow):
        chromosom = [random.randint(0, 1) for _ in range(liczba_genow)]
        populacja.append(chromosom)
    print("\n[INIT] Populacja początkowa:")
    for c in populacja:
        print(c)
    return populacja

In [67]:
# (2) OCENA POPULACJI

""" Ocenę przeprowadza się po kolei dla każdego osobnika (chromosomu) dekodując go do postacji zmiennej x i obliczając dla niej wartość funkcji przystosowania f(x). """

def ocena_populacji(populacja: List[List[int]]) -> List[float]:

    wartosc_przystosowania = []
    for c in populacja:
        x = decode_chromosom(c)
        przystosowanie = funkcja_przystosowania(x)
        wartosc_przystosowania.append(przystosowanie)
    return wartosc_przystosowania

In [4]:
# (3) WARUNEK STOPU

""" Warunek stopu:
 -liczba pokoleń musi być mniejsza niż maksymalna liczba pokoleń
 - małe zróżnicowanie ocen populacji

Warunek stopu możemy związać z oceną bieżącej populacji, na przykład w taki sposób, że procedurę zatrzymamy, jeśli skrajne wartości ocen (najlepsza i najgorsza) będą wystarczająco blisko siebie: 𝑓𝑎 [𝑆(𝑛)] = max 𝑓[𝑥𝑖 (𝑛)] − min 𝑓[𝑥𝑖 (𝑛)] ≤ 𝜀 """

def sprawdzenie_warunku_stopu(wartosc_przystosowania: List[float], pokolenie: int) -> bool:

    if pokolenie >= maksymalnie_pokolen:
        return True

    if max(wartosc_przystosowania) - min(wartosc_przystosowania) < epsilon:
        return True

    return False

In [5]:
# (4) SELEKCJA – METODA RULETKI

""" Selekcja– metoda ruletki. Osobniki wybierane są z prawdopodobieństwami proporcjonalnymi do ich oceny:
Lepsze osobniki mają większą szansę,
ale słabsze nie są całkowicie eliminowane."""

def metoda_ruletki(populacja: List[List[int]],
                             wartosc_przystosowania: List[float]) -> List[List[int]]:

    wyselekcjonowana_populacja = []

    # uniknięcie wartości ujemnych - normalizacja
    min_przystosowanie = min(wartosc_przystosowania)
    skorygowanie_przystosowanie = [f - min_przystosowanie + 1e-6 for f in wartosc_przystosowania]

    # suma ocen
    calkowite_przystosowanie = sum(skorygowanie_przystosowanie)

    # obliczanie prawdopodobieństw
    prawdopodobienstwo = [f / calkowite_przystosowanie for f in skorygowanie_przystosowanie]

    # losowanie k razy - tyle ile jest osobników
    for _ in range(liczba_osobnikow):
        d = random.random() # losowanie liczby d i sprawdzenie w króey przedział wpada
        cumulative = 0
        for i, p in enumerate(prawdopodobienstwo):
            cumulative += p
            if d <= cumulative:
                wyselekcjonowana_populacja.append (populacja[i][:]) #wybór osobnika
                break

    return wyselekcjonowana_populacja

In [6]:
# (5) KRZYŻOWANIE – JEDNOPUNKTOWE

""" Krzyżowanie jednopunktowe - losuje się miejsce punktu krzyżowania, tj. miejsce podziału chromosomów (liczbę naturalną od 1 do r-1), a następnie wymienia między osobnikami fragmenty chromosomów przed miejscem podziału."""

def krzyzowanie_jednopunktowe(rodzic1: List[int],
                            rodzic2: List[int]) -> List[List[int]]:
    if random.random() > prawdopodbienstwo_krzyzowania:
        return rodzic1[:], rodzic2[:]

    point = random.randint(1, liczba_genow - 1)

    dziecko1 = rodzic1[:point] + rodzic2[point:]
    dziecko2 = rodzic2[:point] + rodzic1[point:]

    return dziecko1, dziecko2

In [71]:
# krzyżowanie całej populacji
def krzyzowanie_populacji(populacja: List[List[int]]) -> List[List[int]]:
    potomek = []
    random.shuffle(populacja)

    for i in range(0, liczba_osobnikow, 2):
        rodzic1 = populacja[i]
        rodzic2 = populacja[i + 1]
        dziecko1, dziecko2 = krzyzowanie_jednopunktowe(rodzic1, rodzic2)
        potomek.extend([dziecko1, dziecko2])

    return potomek

In [72]:
# (6) MUTACJA – ZMIANA POJEDYNCZYCH GENÓW

"""" Mutacja – z małym prawdopodobieństwem
zmiana wartości genu na przeciwną."""

def mutacja(chromosom: List[int]) -> List[int]:
    for i in range(len(chromosom)):
        if random.random() <= prawdopodobienstwo_mutacji:
            chromosom[i] = 1 - chromosom[i]
    return chromosom

def mutowanie_populacji(populacja: List[List[int]]) -> List[List[int]]:
    return [mutacja(c[:]) for c in populacja]

In [77]:
# PĘTLA ALGORYTMU GENETYCZNEGO

def algorytm_genetyczny():
    czas_rozpoczecia = time.time()

    populacja = populacja_poczatkowa()
    pokolenie = 0

    while True:
        wartosc_przystosowania = ocena_populacji(populacja)

        print(f"\n[GEN {pokolenie}]")
        for i, c in enumerate(populacja):
            print(f"Osobnik {i}: c={c}, x={decode_chromosom(c)}, f(x)={wartosc_przystosowania[i]:.4f}")

        if sprawdzenie_warunku_stopu(wartosc_przystosowania, pokolenie):
            break

        wyselekcjonowane = metoda_ruletki(populacja, wartosc_przystosowania)
        potomek = krzyzowanie_populacji(wyselekcjonowane)
        populacja = mutowanie_populacji(potomek)

        pokolenie += 1

    wartosc_przystosowania = ocena_populacji(populacja)

    najlepszy_indeks = wartosc_przystosowania.index(max(wartosc_przystosowania))
    najlepszy_chromosom = populacja[najlepszy_indeks]
    najlepszy_x = decode_chromosom(najlepszy_chromosom)
    najlepsze_przystosowanie = wartosc_przystosowania[najlepszy_indeks]

    czas_zakonczenia = time.time()

    print("\n===== WYNIK KOŃCOWY =====")
    print(f"Najlepszy chromosom: {najlepszy_chromosom}")
    print(f"Najlepsze x: {najlepszy_x}")
    print(f"Maksymalna wartość f(x): {najlepsze_przystosowanie:.4f}")
    print(f"Liczba pokoleń: {pokolenie}")
    print(f"Czas obliczeń: {czas_zakonczenia - czas_rozpoczecia:.4f} s")
    return najlepszy_x
# URUCHOMIENIE

if __name__ == "__main__":
    algorytm_genetyczny()


[INIT] Populacja początkowa:
[1, 1, 1, 1, 0, 0, 1, 0, 0, 1]
[0, 0, 1, 1, 1, 0, 0, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 0, 1, 0, 0]
[1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
[1, 1, 0, 0, 1, 1, 0, 1, 1, 1]
[1, 1, 0, 1, 1, 1, 1, 0, 0, 0]
[0, 1, 0, 1, 1, 1, 1, 0, 0, 0]
[0, 1, 0, 1, 1, 1, 0, 0, 0, 0]
[1, 0, 1, 1, 0, 0, 0, 0, 1, 1]
[0, 0, 1, 1, 1, 0, 1, 0, 0, 0]

[GEN 0]
Osobnik 0: c=[1, 1, 1, 1, 0, 0, 1, 0, 0, 1], x=969, f(x)=953.1123
Osobnik 1: c=[0, 0, 1, 1, 1, 0, 0, 1, 1, 1], x=231, f(x)=-230.0030
Osobnik 2: c=[1, 1, 1, 1, 1, 1, 0, 1, 0, 0], x=1012, f(x)=400.7603
Osobnik 3: c=[1, 1, 0, 1, 0, 1, 0, 1, 0, 0], x=852, f(x)=-500.8429
Osobnik 4: c=[1, 1, 0, 0, 1, 1, 0, 1, 1, 1], x=823, f(x)=-79.9313
Osobnik 5: c=[1, 1, 0, 1, 1, 1, 1, 0, 0, 0], x=888, f(x)=779.2612
Osobnik 6: c=[0, 1, 0, 1, 1, 1, 1, 0, 0, 0], x=376, f(x)=-314.5763
Osobnik 7: c=[0, 1, 0, 1, 1, 1, 0, 0, 0, 0], x=368, f(x)=-154.6315
Osobnik 8: c=[1, 0, 1, 1, 0, 0, 0, 0, 1, 1], x=707, f(x)=-99.8140
Osobnik 9: c=[0, 0, 1, 1, 1, 0, 1, 0, 0, 0], x=232, 